<a href="https://colab.research.google.com/github/nurallyssaroslan/KD344403-S2-25-26-G8-PROJECT/blob/main/Milestone4_ModelOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# RandomizedSearchCV is used for broad hyperparameter search.
# It randomly tests a fixed number of parameter combinations from a wide search space.
# This helps reduce runtime while still exploring many possible CatBoost settings.

# Runtime note:
# This RandomizedSearchCV process may take a long time because it tests 100 parameter combinations
# using 7-fold cross-validation, resulting in 700 total model fits.
# In this experiment, it took approximately 2 hours to complete.

# Import tools for hyperparameter tuning and cross-validation
from sklearn.model_selection import RandomizedSearchCV, KFold  # jacq

# Import CatBoost regression model
from catboost import CatBoostRegressor

# Import tqdm to show progress bar in Jupyter Notebook
from tqdm.notebook import tqdm

# Import joblib utilities used by sklearn for parallel processing
from joblib import parallel_backend
import contextlib
import joblib


# Helper function to connect tqdm progress bar with joblib/sklearn parallel tasks
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):

    # This class updates the tqdm progress bar whenever one batch of jobs is completed
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)  # Update progress bar by completed batch size
            return super().__call__(*args, **kwargs)

    # Store the original joblib callback so it can be restored later
    old_batch_callback = joblib.parallel.BatchCompletionCallBack

    # Replace joblib's default callback with the tqdm callback
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback

    try:
        # Run the code inside the context manager with the tqdm progress bar
        yield tqdm_object
    finally:
        # Restore the original joblib callback after the process is finished
        joblib.parallel.BatchCompletionCallBack = old_batch_callback

        # Close the tqdm progress bar
        tqdm_object.close()


# Create the CatBoost Regressor model
cb = CatBoostRegressor(
    verbose=0,                 # Hide CatBoost training output
    random_state=42,           # Ensure reproducible results
    allow_writing_files=False  # Prevent CatBoost from creating extra files/folders
)


# Define the hyperparameter search space for RandomizedSearchCV
param_dist = {
    # Maximum depth of each tree
    "depth": [4, 5, 6, 7, 8, 9, 10, 11, 12],

    # Step size used when updating model weights
    "learning_rate": [0.003, 0.005, 0.01, 0.02, 0.03, 0.05, 0.07, 0.1],

    # Number of boosting iterations/trees
    "iterations": [200, 300, 500, 700, 1000, 1500],

    # L2 regularization strength to reduce overfitting
    "l2_leaf_reg": [1, 3, 5, 7, 9, 12, 15],

    # Fraction of training samples used for each tree
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],

    # Fraction of features used for each tree
    "rsm": [0.6, 0.7, 0.8, 0.9, 1.0]
}


# Set up 7-fold cross-validation
kf = KFold(
    n_splits=7,       # Split training data into 7 parts
    shuffle=True,     # Shuffle data before splitting
    random_state=42   # Ensure the same split every time
)


# Set up Randomized Search for hyperparameter tuning
random_search = RandomizedSearchCV(
    estimator=cb,                    # Model to tune
    param_distributions=param_dist,  # Hyperparameter search space
    n_iter=100,                      # Try 100 random parameter combinations
    cv=kf,                           # Use 7-fold cross-validation
    scoring="r2",                    # Evaluate model using R² score
    n_jobs=-1,                       # Use all available CPU cores
    random_state=42,                 # Ensure reproducible random search
    verbose=0                        # Hide sklearn search output
)


# Total number of model fits = number of random combinations × number of CV folds
total_fits = 100 * 7


# Run RandomizedSearchCV with a progress bar
with tqdm_joblib(tqdm(desc="RandomizedSearchCV Progress", total=total_fits)):
    random_search.fit(X_train, y_train)


# Save the best CatBoost model found by RandomizedSearchCV
best_cb_random = random_search.best_estimator_


# Print the best hyperparameter combination
print("Best Random Search Params:")
print(random_search.best_params_)


# Print the best average cross-validation R² score
print("Best Random Search CV R²:")
print(random_search.best_score_)

In [ ]:
# GridSearchCV is used for focused hyperparameter search.
# It tests every possible combination from a smaller, refined parameter grid.
# This is useful after RandomizedSearchCV has identified promising parameter ranges.

# Runtime note:
# This GridSearchCV process may take a longer time because it tests every possible combination
# in the focused parameter grid using 7-fold cross-validation.
# In this experiment, it took approximately 7 hours to complete.

# Import GridSearchCV for systematic hyperparameter tuning
# Import KFold for cross-validation splitting
from sklearn.model_selection import GridSearchCV, KFold  # jacq

# Import CatBoost regression model
from catboost import CatBoostRegressor

# Import tqdm to show progress bar in Jupyter Notebook
from tqdm.notebook import tqdm

# Import contextlib and joblib to connect tqdm progress bar with sklearn/joblib parallel processing
import contextlib
import joblib

# Import numpy to calculate the total number of parameter combinations
import numpy as np


# Helper function to make tqdm progress bar work with joblib/sklearn
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):

    # This class updates the tqdm progress bar whenever a batch of jobs is completed
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)  # Update progress bar based on completed batch size
            return super().__call__(*args, **kwargs)

    # Store the original joblib callback so it can be restored later
    old_batch_callback = joblib.parallel.BatchCompletionCallBack

    # Replace the original joblib callback with the tqdm callback
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback

    try:
        # Run the code inside this context manager
        yield tqdm_object
    finally:
        # Restore the original joblib callback after GridSearchCV finishes
        joblib.parallel.BatchCompletionCallBack = old_batch_callback

        # Close the progress bar
        tqdm_object.close()


# Create the CatBoost Regressor model for grid search
cb_grid = CatBoostRegressor(
    verbose=0,                 # Hide CatBoost training output
    random_state=42,           # Ensure reproducible results
    allow_writing_files=False  # Prevent CatBoost from creating extra files/folders
)


# Define a focused hyperparameter grid
# This grid is smaller and more specific, usually based on good values found from RandomizedSearchCV
focused_param_grid = {
    # Maximum depth of each decision tree
    "depth": [10, 11, 12],

    # Learning rate controls how much the model updates after each iteration
    "learning_rate": [0.03, 0.05, 0.07],

    # Number of boosting iterations/trees
    "iterations": [600, 700, 800],

    # L2 regularization strength used to reduce overfitting
    "l2_leaf_reg": [2, 3, 5],

    # Fraction of training samples used for each tree
    # Fixed at 0.6 because this value was selected for focused testing
    "subsample": [0.6],

    # Fraction of features used for each tree
    # Fixed at 0.8 because this value was selected for focused testing
    "rsm": [0.8]
}


# Set up 7-fold cross-validation
kf = KFold(
    n_splits=7,       # Split the training data into 7 folds
    shuffle=True,     # Shuffle the data before splitting
    random_state=42   # Ensure the same folds are created every time
)


# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=cb_grid,              # Model to tune
    param_grid=focused_param_grid,  # Hyperparameter grid to test
    cv=kf,                          # Use 7-fold cross-validation
    scoring="r2",                   # Evaluate models using R² score
    n_jobs=2,                       # Use 2 CPU cores for parallel processing
    verbose=0                       # Hide sklearn output
)


# Automatically calculate the number of hyperparameter combinations
# Example: 3 depth values × 3 learning rates × 3 iterations × 3 l2 values × 1 subsample × 1 rsm
total_candidates = np.prod([len(values) for values in focused_param_grid.values()])

# Total fits = number of hyperparameter combinations × number of CV folds
total_fits = total_candidates * kf.get_n_splits()


# Print total candidates and total model fits before training
print(f"Total candidates: {total_candidates}")
print(f"Total fits: {total_fits}")


# Run GridSearchCV with a progress bar
with tqdm_joblib(tqdm(desc="GridSearchCV Progress", total=total_fits)):
    grid_search.fit(X_train, y_train)


# Save the best CatBoost model found by GridSearchCV
best_cb_grid = grid_search.best_estimator_


# Print the best hyperparameter combination
print("Best Grid Search Params:")
print(grid_search.best_params_)


# Print the best average cross-validation R² score
print("Best Grid Search CV R²:")
print(grid_search.best_score_)

In [ ]:
# Tuned Final Results, Run this for Faster Runtime

# Runtime note:
# This final tuned version runs faster because it does not search for parameters again.
# It only evaluates the selected best CatBoost settings using 7-fold cross-validation
# and then tests the final model on unseen test data.

# This final tuned model uses the best hyperparameters selected from the search process.
# RandomizedSearchCV was first used for broad exploration, then GridSearchCV was used for focused fine-tuning.
# Since the best values are already known, this step does not rerun hyperparameter tuning.

# Import KFold for cross-validation
from sklearn.model_selection import KFold  # jacq

# Import evaluation metrics for regression performance
from sklearn.metrics import mean_squared_error, r2_score

# Import CatBoost regression model
from catboost import CatBoostRegressor

# Import numpy to calculate mean, standard deviation, and RMSE
import numpy as np

# Import pandas to create the before-and-after comparison table
import pandas as pd

# Import tqdm to show progress bar in Jupyter Notebook
from tqdm.notebook import tqdm


# Set up 7-fold cross-validation for the final tuned CatBoost model
kf = KFold(
    n_splits=7,       # Split the training data into 7 folds
    shuffle=True,     # Shuffle the data before splitting
    random_state=42   # Ensure the same split every time
)


# Create an empty list to store R² score from each fold
tuned_scores = []


# Loop through each fold in the 7-fold cross-validation
for train_idx, val_idx in tqdm(kf.split(X_train), total=7, desc="Tuned CatBoost 7-Fold CV"):

    # Split the training data into training fold and validation fold
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Create the tuned CatBoost Regressor using the best hyperparameters
    tuned_model = CatBoostRegressor(
        depth=10,                 # Maximum depth of each tree
        iterations=600,           # Number of boosting iterations/trees
        l2_leaf_reg=5,            # L2 regularization strength to reduce overfitting
        learning_rate=0.07,       # Step size for model learning
        rsm=0.8,                  # Fraction of features used for each tree
        subsample=0.6,            # Fraction of training samples used for each tree
        verbose=0,                # Hide CatBoost training output
        random_state=42,          # Ensure reproducible results
        allow_writing_files=False # Prevent CatBoost from creating extra files/folders
    )

    # Train the tuned model using the training fold
    tuned_model.fit(X_tr, y_tr)

    # Evaluate the tuned model on the validation fold using R² score
    score = tuned_model.score(X_val, y_val)

    # Store the R² score for this fold
    tuned_scores.append(score)


# Calculate the average R² score across all 7 folds
tuned_cv_r2 = np.mean(tuned_scores)

# Calculate the standard deviation of R² scores across all 7 folds
# This shows how stable the model performance is across different folds
tuned_cv_std = np.std(tuned_scores)


# Print the tuned cross-validation results
print("Tuned CatBoost CV R² Mean:", tuned_cv_r2)
print("Tuned CatBoost CV R² Std:", tuned_cv_std)


# Final tuned CatBoost evaluation:
# The model is now fitted on the full training set using the same best hyperparameters.
# Then, it is evaluated on the unseen test set.
best_cb_grid = CatBoostRegressor(
    depth=10,
    iterations=600,
    l2_leaf_reg=5,
    learning_rate=0.07,
    rsm=0.8,
    subsample=0.6,
    verbose=0,
    random_state=42,
    allow_writing_files=False
)


# Train the final tuned model on the full training set
best_cb_grid.fit(X_train, y_train)


# Predict target values using the tuned CatBoost model on unseen test data
y_pred_tuned = best_cb_grid.predict(X_test)


# Calculate Mean Squared Error
# MSE measures the average squared difference between actual and predicted values
tuned_mse = mean_squared_error(y_test, y_pred_tuned)


# Calculate Root Mean Squared Error
# RMSE is easier to interpret because it is in the same unit as the target variable
tuned_rmse = np.sqrt(tuned_mse)


# Calculate test R² score
# Test R² shows how well the tuned model performs on unseen data
tuned_r2 = r2_score(y_test, y_pred_tuned)


# Get the original CatBoost result before tuning from the previous result table
catboost_before = df_results[df_results["Model"] == "CatBoost"].iloc[0]


# Calculate the gap between cross-validation R² and test R²
# A small gap suggests that the model generalizes well
tuned_gap = tuned_cv_r2 - tuned_r2


# Determine model status based on CV R², test R², and the CV-test gap
if tuned_cv_r2 < 0.2 and tuned_r2 < 0.2:
    tuned_status = "Underfitting"
elif tuned_gap > 0.05:
    tuned_status = "Overfitting"
else:
    tuned_status = "Good Fit"


# Create a comparison table to show CatBoost performance before and after tuning
comparison_tuning = pd.DataFrame([
    {
        "Model": "CatBoost Before Tuning",
        "CV_R2": catboost_before["CV_R2"],
        "CV_R2_STD": catboost_before["CV_R2_STD"],
        "Test_R2": catboost_before["Test_R2"],
        "MSE": catboost_before["MSE"],
        "RMSE": catboost_before["RMSE"],
        "Gap(CV-Test)": catboost_before["Gap(CV-Test)"],
        "Status": catboost_before["Status"]
    },
    {
        "Model": "CatBoost After Tuning",
        "CV_R2": round(tuned_cv_r2, 4),
        "CV_R2_STD": round(tuned_cv_std, 4),
        "Test_R2": round(tuned_r2, 4),
        "MSE": round(tuned_mse, 4),
        "RMSE": round(tuned_rmse, 4),
        "Gap(CV-Test)": round(tuned_gap, 4),
        "Status": tuned_status
    }
])


# Display the before-and-after tuning comparison table
comparison_tuning